# Evaluación y Validación de Modelos de Machine Learning

## 🎯 Objetivo

Aprender a **evaluar correctamente** modelos de ML y detectar problemas como overfitting y underfitting.

### 📌 Tema Central

> **"Un modelo es tan bueno como su evaluación."**

No basta con entrenar un modelo - necesitamos saber:
* ¿Qué tan bien funciona?
* ¿Funcionará con datos nuevos?
* ¿Qué tipo de errores comete?

---

## 📚 Contenido

1. **Métricas de Evaluación**
   - Clasificación: Accuracy, Precision, Recall, F1, ROC-AUC
   - Regresión: MSE, RMSE, MAE, R²

2. **Validación**
   - Train/Test Split
   - Validación Cruzada (K-Fold)
   - Estratificación

3. **Problemas Comunes**
   - Overfitting (Sobreajuste)
   - Underfitting (Subajuste)
   - Bias-Variance Tradeoff

4. **Curvas de Aprendizaje**
   - Diagnóstico de problemas
   - Soluciones

## Parte 1: Métricas de Clasificación

### 🎯 Matriz de Confusión

Base de todas las métricas de clasificación:

```
                    Predicción
                Positivo  Negativo
Real  Positivo    TP        FN
      Negativo    FP        TN
```

* **TP (True Positive)**: Predijo Positivo, era Positivo ✅
* **TN (True Negative)**: Predijo Negativo, era Negativo ✅
* **FP (False Positive)**: Predijo Positivo, era Negativo ❌ (Error Tipo I)
* **FN (False Negative)**: Predijo Negativo, era Positivo ❌ (Error Tipo II)

---

### 📊 Métricas Principales

#### 1️⃣ **Accuracy (Exactitud)**

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

* **Interpretación**: % de predicciones correctas
* **Cuándo usar**: Clases balanceadas
* **Problema**: Engañosa con clases desbalanceadas

#### 2️⃣ **Precision (Precisión)**

$$\text{Precision} = \frac{TP}{TP + FP}$$

* **Interpretación**: De los que predije positivos, ¿cuántos acerté?
* **Cuándo usar**: Costo alto de FP (ej: spam filtering)

#### 3️⃣ **Recall (Sensibilidad)**

$$\text{Recall} = \frac{TP}{TP + FN}$$

* **Interpretación**: De los positivos reales, ¿cuántos detecté?
* **Cuándo usar**: Costo alto de FN (ej: detección de cáncer)

#### 4️⃣ **F1-Score**

$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

* **Interpretación**: Media armónica de Precision y Recall
* **Cuándo usar**: Balance entre Precision y Recall, clases desbalanceadas

#### 5️⃣ **ROC-AUC**

* **ROC**: Curva TPR vs FPR
* **AUC**: Área bajo la curva (0.5 = aleatorio, 1.0 = perfecto)
* **Cuándo usar**: Comparar modelos, ranking de predicciones

In [0]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

# Dataset de ejemplo: Detección de fraude (desbalanceado)
np.random.seed(42)
n = 1000
X = np.random.randn(n, 5)
y = np.random.binomial(1, 0.05, n)  # 5% fraudes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("╔════════════════════════════════════════════════════════════════════════╗")
print("║              MÉTRICAS DE CLASIFICACIÓN - EJEMPLO                       ║")
print("╚════════════════════════════════════════════════════════════════════════╝")

print(f"\n📊 MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, y_pred))

print(f"\n📊 MÉTRICAS:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")

print(f"\n📊 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

## Parte 2: Validación Cruzada (K-Fold)

### 🎯 El Problema

**Train/Test Split simple**: El resultado puede depender del split aleatorio

### 💡 Solución: K-Fold Cross Validation

```
K = 5 folds

Fold 1: [Test][Train][Train][Train][Train]
Fold 2: [Train][Test][Train][Train][Train]
Fold 3: [Train][Train][Test][Train][Train]
Fold 4: [Train][Train][Train][Test][Train]
Fold 5: [Train][Train][Train][Train][Test]
         ↓
Promedio de 5 scores → Estimación más robusta
```

**Ventajas**:
* ✅ Usa todos los datos para entrenar y evaluar
* ✅ Estimación más robusta del performance
* ✅ Reduce varianza del score

**Stratified K-Fold**:

Mantiene la proporción de clases en cada fold - **crucial para clases desbalanceadas**

### ⚠️ Cuándo NO usar K-Fold:

* **Series temporales**: Usar TimeSeriesSplit (respeta orden temporal)
* **Datos muy grandes**: Costo computacional alto
* **Datos con dependencias**: Cross-validation puede crear data leakage

In [0]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

print("╔════════════════════════════════════════════════════════════════════════╗")
print("║                VALIDACIÓN CRUZADA - EJEMPLO                            ║")
print("╚════════════════════════════════════════════════════════════════════════╝")

model_cv = DecisionTreeClassifier(max_depth=5, random_state=42)

# K-Fold Cross Validation
scores_cv = cross_val_score(model_cv, X, y, cv=5, scoring='accuracy')

print(f"\n📊 K-FOLD (k=5) - Accuracy:")
for i, score in enumerate(scores_cv, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nPromedio: {scores_cv.mean():.4f} (± {scores_cv.std():.4f})")

# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = cross_val_score(model_cv, X, y, cv=skf, scoring='f1')

print(f"\n📊 STRATIFIED K-FOLD (k=5) - F1:")
for i, score in enumerate(scores_skf, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nPromedio: {scores_skf.mean():.4f} (± {scores_skf.std():.4f})")

## Parte 3: Overfitting y Underfitting

### 📉 **Underfitting (Subajuste)**

**Definición**: Modelo demasiado simple

**Síntomas**:
* ❌ Train error alto
* ❌ Test error alto
* ❌ Train ≈ Test (ambos malos)

**Soluciones**:
* ✅ Modelo más complejo
* ✅ Más features
* ✅ Reducir regularización

---

### 📈 **Overfitting (Sobreajuste)**

**Definición**: Modelo memoriza el training set (incluyendo ruido)

**Síntomas**:
* ❌ Train error bajo
* ❌ Test error alto
* ❌ Gran brecha entre train y test

**Soluciones**:
* ✅ Más datos de entrenamiento
* ✅ Regularización (L1, L2, Dropout)
* ✅ Simplificar modelo
* ✅ Early stopping
* ✅ Cross-validation
* ✅ Feature selection

---

### ⚖️ **Bias-Variance Tradeoff**

$$\text{Error Total} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

* **Alto Bias**: Underfitting (modelo muy simple)
* **Alta Variance**: Overfitting (modelo muy complejo)
* **Objetivo**: Balance óptimo

### 📊 Curvas de Aprendizaje

Grafican error vs tamaño del training set:

* **Underfitting**: Ambas curvas altas y paralelas
* **Overfitting**: Gran brecha entre curvas
* **Bueno**: Curvas convergen cerca del 0

In [0]:
from sklearn.tree import DecisionTreeRegressor

print("╔════════════════════════════════════════════════════════════════════════╗")
print("║            OVERFITTING VS UNDERFITTING - DIAGNÓSTICO                   ║")
print("╚════════════════════════════════════════════════════════════════════════╝")

# Dataset sintético
np.random.seed(42)
X_curve = np.sort(np.random.rand(100, 1) * 10, axis=0)
y_curve = np.sin(X_curve).ravel() + np.random.randn(100) * 0.5

X_tr, X_te, y_tr, y_te = train_test_split(X_curve, y_curve, test_size=0.3, random_state=42)

models = {
    'Underfitting (depth=1)': DecisionTreeRegressor(max_depth=1),
    'Bueno (depth=5)': DecisionTreeRegressor(max_depth=5),
    'Overfitting (depth=20)': DecisionTreeRegressor(max_depth=20)
}

from sklearn.metrics import mean_squared_error

for name, model in models.items():
    model.fit(X_tr, y_tr)
    train_score = mean_squared_error(y_tr, model.predict(X_tr))
    test_score = mean_squared_error(y_te, model.predict(X_te))
    
    print(f"\n{name}:")
    print(f"  Train MSE: {train_score:.4f}")
    print(f"  Test MSE:  {test_score:.4f}")
    print(f"  Brecha:    {abs(test_score - train_score):.4f}")

## 📍d Conclusiones y Mejores Prácticas

### 🎯 Key Takeaways

1. **La métrica importa**
   - Clasificación: F1 para desbalance, ROC-AUC para ranking
   - Regresión: RMSE (penaliza outliers), MAE (robusto)
   - Nunca solo Accuracy en clases desbalanceadas

2. **Validación es crítica**
   - Siempre usar train/test split
   - K-Fold CV para estimación robusta
   - Stratified para clases desbalanceadas

3. **Overfitting es el enemigo #1**
   - Usar regularización
   - Más datos siempre ayuda
   - Cross-validation para detectarlo

4. **No confiar en una sola métrica**
   - Mirar múltiples métricas
   - Analizar matriz de confusión
   - Entender tipo de errores

### ✅ Checklist de Evaluación:

- [ ] Train/test split realizado (o K-Fold CV)
- [ ] Métricas apropiadas para el problema
- [ ] Validación con datos no vistos
- [ ] Curvas de aprendizaje revisadas
- [ ] Overfitting/underfitting descartado
- [ ] Errores analizados (matriz de confusión)
- [ ] Performance comparado con baseline

### 🚨 Errores Comunes:

* ❌ Evaluar solo en training set
* ❌ Usar Accuracy en clases desbalanceadas
* ❌ No estratificar en CV con desbalance
* ❌ Optimizar una métrica sin mirar otras
* ❌ No tener baseline para comparar
* ❌ Ajustar hiperparámetros en test set

### 📚 Recursos:

* **Práctica**: Kaggle competitions - estudiar métricas usadas
* **Herramientas**: scikit-learn metrics, MLflow tracking
* **Visualización**: Confusion matrix, ROC curves, learning curves